In [1]:
import pandas as pd
import numpy as  np
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import matplotlib.pyplot as plt
import datetime as dt
from calendar import day_name

In [3]:
user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"

password_encoded = quote_plus(password)

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
)

start_date = '2025-09-01'
end_date = '2025-09-09'
campaigns = (1605, 1553, 1025, 1700)


query = text(f"""

WITH base AS (
    SELECT DISTINCT
        asrd.agentid,
        u.username AS agent,
        asrd.agentstaterawdataid,
        ias.description AS agent_status_description,
        CASE
            WHEN ias.description NOT IN ('Pause') AND r.description IS NULL THEN 'Atendendo'
            WHEN r.description = 'Atendimento Blip' THEN 'Atendimento Blip'
            WHEN r.description = 'Tabulacao' THEN 'Tabulacao'
            ELSE r.description
        END AS reason_description,
        (asrd.endstate - asrd.startstate) AS pause_duration,
        asrd.startstate,
        asrd.endstate
    FROM app_olos.agent_state_raw_data asrd
    LEFT JOIN app_olos.users u
        ON asrd.agentid = u.agentid
    LEFT JOIN app_olos.reason r
        ON r.reasonid = asrd.reason
    LEFT JOIN app_olos.info_agent_status ias
        ON ias.agentstatusid = asrd.agentstatus
    WHERE
        asrd.startstate BETWEEN '2026-03-01' AND '2026-03-31'
        AND ias.description NOT LIKE '%Other%'
        AND asrd.campaignid IN ('1562','1025','1605','1299')
),

tb AS (
    SELECT
        agentid,
        agent,
        agent_status_description,
        startstate::date AS data,
        reason_description,

        -- total em segundos (ótimo pro Power BI)
        EXTRACT(EPOCH FROM SUM(pause_duration)) AS total_inserttime_segundos,

        -- total formatado HH:MM:SS (visual)
        TO_CHAR(SUM(pause_duration), 'HH24:MI:SS') AS total_inserttime,

        MIN(startstate) AS pauses_start,
        MAX(endstate) AS pauses_end
    FROM base
    GROUP BY
        agentid,
        agent,
        agent_status_description,
        startstate::date,
        reason_description
),

logs AS (
    SELECT
        agent,
        data,
        MIN(log_in)::time  AS log_in,
        MAX(log_out)::time AS log_out
    FROM (
        SELECT
            u.username AS agent,
            asrd.startstate::date AS data,
            MIN(asrd.startstate::time) AS log_in,
            MAX(asrd.endstate::time)   AS log_out
        FROM app_olos.agent_state_raw_data asrd
        LEFT JOIN app_olos.users u
            ON asrd.agentid = u.agentid
        WHERE
            asrd.startstate BETWEEN '2026-03-01' AND '2026-03-31'
            AND asrd.campaignid IN ('1562','1025','1605','1299')
        GROUP BY
            u.username,
            asrd.startstate::date
    ) x
    GROUP BY agent, data
)

SELECT
    tb.*,
    logs.log_in,
    logs.log_out
FROM tb
LEFT JOIN logs
    ON logs.agent = tb.agent
    AND logs.data  = tb.data
ORDER BY tb.agent, tb.data, tb.reason_description;
    
""")

try:
    df = pd.read_sql(query, engine)
    num_linhas = df.shape[0]
    print(f"Consulta executada com sucesso! Retornou {num_linhas} linhas.")
    hoje = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"Data e hora da execução: {hoje}")
except Exception as e:
    df = pd.read_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\Construtores\backup.xlsx')
    
    print("❌ Erro ao executar a consulta:")
    print(f"➡️ {e}")
    print(f'Lendo arquivo auxiliar...')

Consulta executada com sucesso! Retornou 2064 linhas.
Data e hora da execução: 2026-03-26 11:50:06


In [4]:
df

,agentid,agent,agent_status_description,data,reason_description,total_inserttime_segundos,total_inserttime,pauses_start,pauses_end,log_in,log_out
0,6216,Camila Schmidt da Silva,Pause,2026-03-02,20 minutos NR17,2396.706,00:39:56,2026-03-02 12:28:15.607,2026-03-02 13:08:12.313,09:23:49.010000,16:17:46.487000
1,6216,Camila Schmidt da Silva,Ending,2026-03-02,Atendendo,0.030,00:00:00,2026-03-02 16:17:46.457,2026-03-02 16:17:46.487,09:23:49.010000,16:17:46.487000
2,6216,Camila Schmidt da Silva,Pausing,2026-03-02,Atendendo,0.717,00:00:00,2026-03-02 10:48:43.140,2026-03-02 13:08:19.533,09:23:49.010000,16:17:46.487000
3,6216,Camila Schmidt da Silva,Wrap,2026-03-02,Atendendo,7255.028,02:00:55,2026-03-02 09:39:54.157,2026-03-02 16:17:42.237,09:23:49.010000,16:17:46.487000
4,6216,Camila Schmidt da Silva,Idle,2026-03-02,Atendendo,8438.229,02:20:38,2026-03-02 09:23:49.010,2026-03-02 16:17:46.457,09:23:49.010000,16:17:46.487000
...,...,...,...,...,...,...,...,...,...,...,...
2059,3293,Vanessa Correia,Pausing,2026-03-26,Atendendo,0.644,00:00:00,2026-03-26 09:22:54.070,2026-03-26 10:32:41.040,09:05:02.883000,10:32:41.040000
2060,3293,Vanessa Correia,Talking,2026-03-26,Atendendo,650.476,00:10:50,2026-03-26 09:18:39.477,2026-03-26 10:28:59.447,09:05:02.883000,10:32:41.040000
2061,3293,Vanessa Correia,Pause,2026-03-26,Atendimento Blip,54.610,00:00:54,2026-03-26 09:22:54.340,2026-03-26 09:23:48.950,09:05:02.883000,10:32:41.040000
2062,3293,Vanessa Correia,Pause,2026-03-26,Atendimento Voll,54.610,00:00:54,2026-03-26 09:22:54.340,2026-03-26 09:23:48.950,09:05:02.883000,10:32:41.040000


In [6]:
depara = {
'Abertura diaria':'Abertura diaria',
'Ajuste de ponto':'Ajuste de ponto',
'Pausa - Ajuste Ponto':'Ajuste de ponto',
'Atendendo':'Em Atendimento',
'Atendimento Blip':'Atendimento Blip',
'Atendimento Voll':'Atendimento Blip',
'Pausa Whatsapp':'Atendimento Blip',
'Banheiro':'Banheiro',
'Banheiro_workforce':'Banheiro',
'Feedback':'Feedback',
'Pausa cadastro':'Pausa cadastro',
'10 minutos NR17':'Pausa NR',
'20 minutos NR17':'Pausa NR',
'Almoco':'Pausa NR',
'Hora Extra':'Pausa NR',
'Pausa almoco':'Pausa NR',
'Pausa Descanso 1':'Pausa NR',
'Pausa Descanso 2':'Pausa NR',
'Pausa 20':'Pausa NR',
'2 Intervalo':'Pausa NR',
'Pausa - Pessoal':'Pausa Pessoal',
'Pausa Pessoal':'Pausa Pessoal',
'Problemas Sistemicos':'Pausa Pessoal',
'Pausa Prelecao':'Pausa Prelecao',
'Reuniao':'Reuniao',
'Saida Particular':'Saida Particular',
'Treinamento':'Treinamento',
}

In [26]:
df['Pausa_corrigida'] = df['reason_description'].map(depara)
df['total_inserttime'] = pd.to_timedelta(df['total_inserttime'])
df

,agentid,agent,agent_status_description,data,reason_description,total_inserttime_segundos,total_inserttime,pauses_start,pauses_end,log_in,log_out,Pausa_corrigida
0,6216,Camila Schmidt da Silva,Pause,2026-03-02,20 minutos NR17,2396.706,0 days 00:39:56,2026-03-02 12:28:15.607,2026-03-02 13:08:12.313,09:23:49.010000,16:17:46.487000,Pausa NR
1,6216,Camila Schmidt da Silva,Ending,2026-03-02,Atendendo,0.030,0 days 00:00:00,2026-03-02 16:17:46.457,2026-03-02 16:17:46.487,09:23:49.010000,16:17:46.487000,Em Atendimento
2,6216,Camila Schmidt da Silva,Pausing,2026-03-02,Atendendo,0.717,0 days 00:00:00,2026-03-02 10:48:43.140,2026-03-02 13:08:19.533,09:23:49.010000,16:17:46.487000,Em Atendimento
3,6216,Camila Schmidt da Silva,Wrap,2026-03-02,Atendendo,7255.028,0 days 02:00:55,2026-03-02 09:39:54.157,2026-03-02 16:17:42.237,09:23:49.010000,16:17:46.487000,Em Atendimento
4,6216,Camila Schmidt da Silva,Idle,2026-03-02,Atendendo,8438.229,0 days 02:20:38,2026-03-02 09:23:49.010,2026-03-02 16:17:46.457,09:23:49.010000,16:17:46.487000,Em Atendimento
...,...,...,...,...,...,...,...,...,...,...,...,...
2059,3293,Vanessa Correia,Pausing,2026-03-26,Atendendo,0.644,0 days 00:00:00,2026-03-26 09:22:54.070,2026-03-26 10:32:41.040,09:05:02.883000,10:32:41.040000,Em Atendimento
2060,3293,Vanessa Correia,Talking,2026-03-26,Atendendo,650.476,0 days 00:10:50,2026-03-26 09:18:39.477,2026-03-26 10:28:59.447,09:05:02.883000,10:32:41.040000,Em Atendimento
2061,3293,Vanessa Correia,Pause,2026-03-26,Atendimento Blip,54.610,0 days 00:00:54,2026-03-26 09:22:54.340,2026-03-26 09:23:48.950,09:05:02.883000,10:32:41.040000,Atendimento Blip
2062,3293,Vanessa Correia,Pause,2026-03-26,Atendimento Voll,54.610,0 days 00:00:54,2026-03-26 09:22:54.340,2026-03-26 09:23:48.950,09:05:02.883000,10:32:41.040000,Atendimento Blip


In [68]:
pausas = (
    df.pivot_table(
        index=['data'],
        columns='Pausa_corrigida',
        values=('total_inserttime'),
        aggfunc='mean'
    )
).reset_index()
pausas = pausas.fillna(0)
pausas = pausas.applymap(
    lambda x: str(x).split(' ')[-1] if pd.notnull(x) else x
)
for col in pausas.columns:
    if pausas[col].dtype == 'timedelta64[ns]':
        pausas[col] = pausas[col].dt.round('s').dt.strtime('%H:%M:%S')


C:\Users\wconceicao\AppData\Local\Temp\ipykernel_14060\3093848486.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pausas = pausas.applymap(


In [69]:

pausas


Pausa_corrigida,data,Abertura diaria,Ajuste de ponto,Atendimento Blip,Banheiro,Em Atendimento,Pausa NR,Pausa Pessoal,Pausa cadastro,Treinamento
0,2026-03-02,0,00:08:01.333333333,01:43:24.714285714,00:13:22.375000,00:36:48.038461538,00:25:17.230769230,00:08:01.333333333,00:02:34,00:10:51.500000
1,2026-03-03,0,00:05:44,01:30:02,00:13:09.285714285,00:27:54.350877192,00:27:14.833333333,00:05:44,0,00:45:07
2,2026-03-04,0,00:18:01.666666666,02:10:57.750000,00:13:45.888888888,00:27:49.473684210,00:25:49.666666666,00:18:01.666666666,00:00:00,00:33:19
3,2026-03-05,00:00:02,00:16:17.500000,01:34:51.900000,00:11:55.200000,00:32:27.107142857,00:24:39.285714285,00:16:17.500000,0,03:58:18.333333333
4,2026-03-06,00:15:10,00:35:31.500000,02:06:32.250000,00:19:30.250000,00:30:40.566037735,00:26:04.142857142,00:35:31.500000,0,02:50:40.500000
5,2026-03-07,00:02:12,00:01:44,01:39:08.125000,00:12:24.833333333,00:30:37.511111111,00:25:47.500000,00:01:44,0,00:06:47
6,2026-03-09,0,00:14:16.500000,01:41:15.727272727,00:09:44.555555555,00:27:54.698412698,00:28:47.937500,00:14:16.500000,0,00:44:35.111111111
7,2026-03-10,0,00:39:35.500000,01:02:51.363636363,00:10:25.600000,00:33:04.731343283,00:24:03.923076923,00:39:35.500000,00:17:10,00:24:05.600000
8,2026-03-11,00:06:20,00:13:20,01:06:42.909090909,00:10:30.200000,00:25:17.671875,00:24:21,00:13:20,0,00:01:45
9,2026-03-12,0,00:16:04.250000,01:34:41.181818181,00:12:30.375000,00:27:45.681818181,00:19:51.062500,00:16:04.250000,00:02:17,00:23:09.500000
